# Voxel — train one AI voice

Runtime → Change runtime type → **T4 GPU**, then Runtime → **Run all**. Fill in the three settings below first. Everything is saved to your Google Drive under `Voxel/`, so if Colab disconnects you just run it again and it continues.

In [ ]:
#@title Settings
VOICE_NAME = "woman-1"  #@param {type:"string"}
REFERENCE = "F:0"  #@param {type:"string"}
HOURS_OF_SPEECH = 20  #@param {type:"number"}
TRAIN_STEPS = 100000  #@param {type:"integer"}
# REFERENCE = whose voice to learn:
#   "F:0" / "M:3"  -> the 1st female / 4th male reader of the free corpus (change the number for another person)
#   "1088"         -> a LibriSpeech speaker id
#   "https://..."  -> a direct link to a 10-30 s clean wav/mp3 of any voice

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
ROOT = '/content/drive/MyDrive/Voxel'
RUN = f'{ROOT}/runs/{VOICE_NAME}'
os.makedirs(f'{ROOT}/references', exist_ok=True)
os.makedirs(RUN, exist_ok=True)
print('run folder:', RUN)

In [ ]:
%%bash
set -e
mkdir -p /content/work && cd /content/work
[ -d LLVC ] || git clone --depth 1 https://github.com/KoeAI/LLVC.git
sed -i 's/from scipy.signal import kaiser/from scipy.signal.windows import kaiser/' LLVC/minimal_quickvc/pqmf.py
pip -q install soundfile auraloss librosa huggingface_hub onnx onnxscript speechbrain
mkdir -p train
echo ok

In [ ]:
%%writefile /content/work/train/make_pairs.py
"""Build LLVC training pairs with QuickVC as the teacher.

For every source clip: cut it into fixed-length pieces, convert each piece into the target
voice (given by one reference clip), and write <id>_original.wav / <id>_converted.wav at
16 kHz into train/ val/ dev/ folders, the layout LLVC's dataset.py expects.

Usage: python make_pairs.py --src <folder of wav/flac> --ref <reference.wav> --out <dataset dir>
                            [--hours 5] [--piece 65536] [--device cuda]
"""
import argparse
import glob
import os
import random
import sys

import numpy as np
import soundfile as sf
import torch
import torchaudio.functional as AF
from tqdm import tqdm

HERE = os.path.dirname(os.path.abspath(__file__))
LLVC_DIR = os.path.join(os.path.dirname(HERE), "LLVC")
sys.path.insert(0, LLVC_DIR)
from mel_processing import mel_spectrogram_torch  # noqa: E402
from minimal_quickvc.models import SynthesizerTrn  # noqa: E402
from minimal_quickvc.utils import load_checkpoint  # noqa: E402

SR = 16000


def load_wav(path):
    audio, sr = sf.read(path, dtype="float32", always_2d=True)
    audio = torch.from_numpy(audio.mean(axis=1))
    if sr != SR:
        audio = AF.resample(audio, sr, SR)
    return audio


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--src", required=True)
    ap.add_argument("--ref", required=True)
    ap.add_argument("--out", required=True)
    ap.add_argument("--hours", type=float, default=5.0, help="how much source audio to use")
    ap.add_argument("--piece", type=int, default=65536, help="samples per training piece (LLVC wav_len)")
    ap.add_argument("--device", default="cuda" if torch.cuda.is_available() else "cpu")
    ap.add_argument("--quickvc", default=os.path.join(LLVC_DIR, "llvc_models", "models", "quickvc", "quickvc_100.pth"))
    ap.add_argument("--seed", type=int, default=1234)
    args = ap.parse_args()

    random.seed(args.seed)
    device = torch.device(args.device)
    hubert = torch.hub.load("bshall/hubert:main", "hubert_soft", trust_repo=True).eval().to(device)
    model = SynthesizerTrn().eval()
    load_checkpoint(args.quickvc, model, None)
    model.to(device)
    mel = mel_spectrogram_torch(load_wav(args.ref).unsqueeze(0), SR, 1280, 80, 320, 1280, 0, None).to(device)

    files = sorted(glob.glob(os.path.join(args.src, "**", "*.wav"), recursive=True)
                   + glob.glob(os.path.join(args.src, "**", "*.flac"), recursive=True))
    random.shuffle(files)
    budget = int(args.hours * 3600 * SR)
    for split in ("train", "val", "dev"):
        os.makedirs(os.path.join(args.out, split), exist_ok=True)

    written = 0
    used = 0
    pbar = tqdm(total=budget, unit="samples", unit_scale=True)
    with torch.inference_mode():
        for f in files:
            if used >= budget:
                break
            audio = load_wav(f)
            # Drop the last partial piece; pad nothing, so every pair is exactly `piece` long.
            n = len(audio) // args.piece
            for k in range(n):
                piece = audio[k * args.piece:(k + 1) * args.piece]
                units = hubert.units(piece.view(1, 1, -1).to(device)).transpose(2, 1)
                converted = model.infer(units, mel=mel).reshape(-1).cpu()
                converted = converted[:len(piece)]
                if len(converted) < len(piece):
                    converted = torch.cat([converted, torch.zeros(len(piece) - len(converted))])
                # 1% val, 1% dev, rest train
                r = random.random()
                split = "val" if r < 0.01 else "dev" if r < 0.02 else "train"
                stem = os.path.join(args.out, split, f"{written:06d}")
                sf.write(stem + "_original.wav", (piece.numpy() * 32767).astype(np.int16), SR)
                sf.write(stem + "_converted.wav", (converted.clamp(-1, 1).numpy() * 32767).astype(np.int16), SR)
                written += 1
                used += len(piece)
                pbar.update(len(piece))
                if used >= budget:
                    break
    pbar.close()
    print(f"wrote {written} pairs ({used / SR / 3600:.2f} h) to {args.out}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/work/train/train_llvc.py
"""Train an LLVC voice from QuickVC-made pairs. Single process, resumable, CPU or GPU.

A cleaned-up version of Koe AI's train.py: same model, same losses (multi-period
discriminator + feature matching + multi-resolution mel), minus the fairseq/HuBERT
perceptual loss so nothing exotic has to be installed. Checkpoints land in <dir>/checkpoints
and training resumes from the newest one automatically.

Usage: python train_llvc.py --dir <run dir containing config.json> [--steps 200000]
"""
import argparse
import glob
import json
import os
import random
import sys
import time

import soundfile as sf
import torch
import torch.nn.functional as F

HERE = os.path.dirname(os.path.abspath(__file__))
LLVC_DIR = os.path.join(os.path.dirname(HERE), "LLVC")
sys.path.insert(0, LLVC_DIR)
from dataset import LLVCDataset  # noqa: E402
from discriminators import MultiPeriodDiscriminator, discriminator_loss, feature_loss, generator_loss  # noqa: E402
from model import Net  # noqa: E402
import utils  # noqa: E402


def latest(dirpath, pattern):
    files = glob.glob(os.path.join(dirpath, pattern))
    if not files:
        return None
    return max(files, key=lambda p: int(os.path.basename(p).split("_")[1].split(".")[0]))


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--dir", required=True)
    ap.add_argument("--steps", type=int, default=200000)
    ap.add_argument("--device", default="cuda" if torch.cuda.is_available() else "cpu")
    args = ap.parse_args()

    config = json.load(open(os.path.join(args.dir, "config.json")))
    device = torch.device(args.device)
    torch.manual_seed(config["seed"])
    random.seed(config["seed"])
    ckpt_dir = os.path.join(args.dir, "checkpoints")
    samples_dir = os.path.join(args.dir, "samples")
    os.makedirs(ckpt_dir, exist_ok=True)
    os.makedirs(samples_dir, exist_ok=True)

    data = LLVCDataset(**config["data"], dset="train")
    print(f"{len(data)} training pairs")
    loader = torch.utils.data.DataLoader(data, batch_size=config["batch_size"], shuffle=True, drop_last=True,
                                         num_workers=2 if device.type == "cuda" else 0)

    net_g = Net(**config["model_params"]).to(device)
    net_d = MultiPeriodDiscriminator(periods=config["periods"]).to(device)
    print(f"generator {utils.model_size(net_g):.2f} M params")
    opt = config["optim"]
    optim_g = torch.optim.AdamW(net_g.parameters(), opt["lr"], betas=opt["betas"], eps=opt["eps"], weight_decay=opt["weight_decay"])
    optim_d = torch.optim.AdamW(net_d.parameters(), opt["lr"], betas=opt["betas"], eps=opt["eps"], weight_decay=opt["weight_decay"])

    step, epoch = 0, 0
    g_path, d_path = latest(ckpt_dir, "G_*.pth"), latest(ckpt_dir, "D_*.pth")
    if g_path and d_path:
        net_g, optim_g, _, epoch, step = utils.load_checkpoint(g_path, net_g, optim_g)
        net_d, optim_d, _, _, _ = utils.load_checkpoint(d_path, net_d, optim_d)
        print(f"resumed from step {step}")
    sched_g = torch.optim.lr_scheduler.ExponentialLR(optim_g, gamma=config["lr_sched"]["lr_decay"], last_epoch=epoch - 1)
    sched_d = torch.optim.lr_scheduler.ExponentialLR(optim_d, gamma=config["lr_sched"]["lr_decay"], last_epoch=epoch - 1)

    use_amp = config.get("fp16_run", False) and device.type == "cuda"
    scaler = torch.amp.GradScaler(enabled=use_amp)
    test_wavs = [(os.path.basename(p), utils.load_wav_to_torch(p, config["data"]["sr"]))
                 for p in sorted(glob.glob(os.path.join(config["test_dir"], "*.wav")))[:2]]
    seg = config["segment_size"]
    t0 = time.time()

    while step < args.steps:
        net_g.train(); net_d.train()
        for og, gt in loader:
            if step >= args.steps:
                break
            og, gt = og.to(device), gt.to(device)

            with torch.autocast(device.type, enabled=use_amp):
                output = net_g(og)
            # Random slice for the discriminator (cheaper), full length for mel loss.
            if seg < output.shape[-1]:
                s = random.randint(0, output.shape[-1] - seg - 1)
                gt_s, out_s = gt[:, :, s:s + seg], output.detach()[:, :, s:s + seg]
            else:
                gt_s, out_s = gt, output.detach()

            with torch.autocast(device.type, enabled=use_amp):
                d_r, d_g, _, _ = net_d(out_s, gt_s)
            loss_d, _, _ = discriminator_loss(d_r, d_g)
            optim_d.zero_grad(set_to_none=True)
            scaler.scale(loss_d).backward()
            scaler.unscale_(optim_d)
            torch.nn.utils.clip_grad_norm_(net_d.parameters(), config["grad_clip_threshold"])
            scaler.step(optim_d)

            with torch.autocast(device.type, enabled=use_amp):
                d_r, d_g, fmap_r, fmap_g = net_d(gt, output)
            loss_mel = utils.aux_mel_loss(output.float(), gt, config) * config["aux_mel"]["c"]
            loss_fm = feature_loss(fmap_r, fmap_g) * config["feature_loss_c"]
            loss_gen, _ = generator_loss(d_g)
            loss_g = loss_gen * config["disc_loss_c"] + loss_fm + loss_mel
            optim_g.zero_grad(set_to_none=True)
            scaler.scale(loss_g).backward()
            scaler.unscale_(optim_g)
            torch.nn.utils.clip_grad_norm_(net_g.parameters(), config["grad_clip_threshold"])
            scaler.step(optim_g)
            scaler.update()
            step += 1

            if step % config["log_interval"] == 0 or step == 1:
                rate = step / max(time.time() - t0, 1e-6)
                print(f"step {step}  mel {loss_mel.item():.2f}  fm {loss_fm.item():.2f}  gen {loss_gen.item():.2f}  "
                      f"disc {loss_d.item():.2f}  ({rate:.2f} steps/s)", flush=True)
            if step % config["checkpoint_interval"] == 0:
                utils.save_state(net_g, optim_g, optim_g.param_groups[0]["lr"], epoch, step, os.path.join(ckpt_dir, f"G_{step}.pth"))
                utils.save_state(net_d, optim_d, optim_d.param_groups[0]["lr"], epoch, step, os.path.join(ckpt_dir, f"D_{step}.pth"))
                # Keep only the two newest checkpoints: they're ~200 MB each with optimizer state.
                for pat in ("G_*.pth", "D_*.pth"):
                    old = sorted(glob.glob(os.path.join(ckpt_dir, pat)), key=lambda p: int(os.path.basename(p).split("_")[1].split(".")[0]))
                    for p in old[:-2]:
                        os.remove(p)
                net_g.eval()
                with torch.no_grad():
                    for name, wav in test_wavs:
                        out = net_g(wav.view(1, 1, -1).to(device)).reshape(-1).cpu().clamp(-1, 1)
                        sf.write(os.path.join(samples_dir, f"{step}_{name}"), out.numpy(), config["data"]["sr"])
                net_g.train()
                print(f"saved checkpoint at step {step}", flush=True)
        epoch += 1
        sched_g.step(); sched_d.step()

    utils.save_state(net_g, optim_g, optim_g.param_groups[0]["lr"], epoch, step, os.path.join(ckpt_dir, f"G_{step}.pth"))
    utils.save_state(net_d, optim_d, optim_d.param_groups[0]["lr"], epoch, step, os.path.join(ckpt_dir, f"D_{step}.pth"))
    print(f"done at step {step}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/work/train/export_onnx.py
"""Export a trained LLVC checkpoint to the ONNX + JSON pair Voxel loads.

Usage: python export_onnx.py --checkpoint G_123.pth --config config.json --out <name>.onnx [--chunk-factor 4]
"""
import argparse
import json
import os
import sys

import torch

HERE = os.path.dirname(os.path.abspath(__file__))
LLVC_DIR = os.path.join(os.path.dirname(HERE), "LLVC")
sys.path.insert(0, LLVC_DIR)
from model import Net  # noqa: E402


class StreamStep(torch.nn.Module):
    def __init__(self, net):
        super().__init__()
        self.net = net

    def forward(self, chunk, enc_buf, dec_buf, out_buf, pre_ctx):
        return self.net(chunk, enc_buf.clone(), dec_buf.clone(), out_buf.clone(), pre_ctx.clone(), pad=False)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--checkpoint", required=True)
    ap.add_argument("--config", required=True)
    ap.add_argument("--out", required=True)
    ap.add_argument("--chunk-factor", type=int, default=4)
    args = ap.parse_args()

    config = json.load(open(args.config))
    model = Net(**config["model_params"])
    model.load_state_dict(torch.load(args.checkpoint, map_location="cpu")["model"])
    model.eval()
    L = model.L
    chunk_len = model.dec_chunk_size * L * args.chunk_factor
    enc_buf, dec_buf, out_buf = model.init_buffers(1, torch.device("cpu"))
    pre_ctx = model.convnet_pre.init_ctx_buf(1, torch.device("cpu"))
    chunk = torch.zeros(1, 1, 2 * L + chunk_len)

    with torch.inference_mode():
        torch.onnx.export(StreamStep(model).eval(), (chunk, enc_buf, dec_buf, out_buf, pre_ctx), args.out,
                          input_names=["chunk", "enc_buf", "dec_buf", "out_buf", "pre_ctx"],
                          output_names=["out", "enc_buf_out", "dec_buf_out", "out_buf_out", "pre_ctx_out"],
                          opset_version=17, dynamo=False)
    meta = {"sample_rate": config["data"]["sr"], "L": L, "chunk_len": chunk_len, "context": 2 * L,
            "enc_buf": list(enc_buf.shape), "dec_buf": list(dec_buf.shape),
            "out_buf": list(out_buf.shape), "pre_ctx": list(pre_ctx.shape)}
    json.dump(meta, open(os.path.splitext(args.out)[0] + ".json", "w"), indent=2)
    print(f"exported {args.out} ({os.path.getsize(args.out) // 1024} KB), chunk {chunk_len} samples")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/work/train/config_template.json
{
  "data": { "dir": "DATASET_DIR", "sr": 16000, "wav_len": 65536 },
  "model_params": {
    "label_len": 1, "L": 16, "enc_dim": 512, "num_enc_layers": 8, "dec_dim": 256, "num_dec_layers": 1,
    "dec_buf_len": 13, "dec_chunk_size": 13, "out_buf_len": 4, "use_pos_enc": true, "decoder_dropout": 0.1,
    "convnet_config": {
      "convnet_prenet": true,
      "out_channels": [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
      "kernel_sizes": [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3],
      "dilations": [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
      "dropout": 0.5, "combine_residuals": null, "skip_connection": "add", "use_residual_blocks": true
    }
  },
  "optim": { "lr": 5e-4, "betas": [0.8, 0.999], "eps": 1e-9, "weight_decay": 0.0 },
  "lr_sched": { "lr_decay": 0.999 },
  "batch_size": 9,
  "fp16_run": true,
  "segment_size": 10240,
  "log_interval": 200,
  "checkpoint_interval": 2000,
  "aux_mel": {
    "type": "multires", "c": 10,
    "params": { "num_mels": 80, "hop_size": [120, 240, 50], "win_size": [600, 1200, 240], "n_fft": [1024, 2048, 512], "fmin": 0.0, "fmax": null }
  },
  "disc_loss_c": 1,
  "feature_loss_c": 2,
  "seed": 1234,
  "test_dir": "TEST_DIR",
  "periods": [2, 3, 5, 7, 11, 17, 23, 37],
  "grad_clip_threshold": 1.0
}


In [ ]:
# QuickVC teacher + HuBERT (cached in the session)
from huggingface_hub import hf_hub_download
hf_hub_download('KoeAI/llvc', 'models/quickvc/quickvc_100.pth', local_dir='/content/work/LLVC/llvc_models')
import torch
torch.hub.load('bshall/hubert:main', 'hubert_soft', trust_repo=True)
print('teacher ready')

In [ ]:
%%bash
# Free speech corpus to learn from (LibriSpeech, CC-BY 4.0). ~6 GB, cached on Drive.
# Verifies the archive size and the number of extracted files, so an interrupted session can't leave it half done.
set -e
D=/content/drive/MyDrive/Voxel/corpus; mkdir -p $D; T=$D/train-clean-100.tar.gz
size() { stat -c %s "$1" 2>/dev/null || echo 0; }
if [ "$(size $T)" -lt 6000000000 ]; then echo 'downloading corpus (6.3 GB, once)'; rm -f $T; wget -q --show-progress -O $T https://www.openslr.org/resources/12/train-clean-100.tar.gz; fi
echo "archive: $(size $T) bytes"
mkdir -p /content/corpus && cd /content/corpus
N=$(find LibriSpeech/train-clean-100 -name '*.flac' 2>/dev/null | wc -l)
if [ "$N" -lt 28000 ]; then echo "extracting ($N of 28539 files present)"; rm -rf LibriSpeech; tar -xzf $T; fi
echo "corpus ready: $(find LibriSpeech/train-clean-100 -name '*.flac' | wc -l) files"

In [ ]:
# Reference clip of the target voice (about 30 s at 16 kHz)
import glob, os, re, urllib.request
import numpy as np, soundfile as sf, torch, torchaudio.functional as AF
ref = f'{ROOT}/references/{VOICE_NAME}.wav'
CORPUS = '/content/corpus/LibriSpeech/train-clean-100'

def to16k(path):
    try:
        a, sr = sf.read(path, dtype='float32', always_2d=True); a = a.mean(axis=1)
    except Exception:
        import librosa
        a, sr = librosa.load(path, sr=None, mono=True)
    a = torch.from_numpy(a)
    return AF.resample(a, sr, 16000) if sr != 16000 else a

def speakers_by_sex(sex):
    ids = []
    for line in open('/content/corpus/LibriSpeech/SPEAKERS.TXT'):
        if line.startswith(';'): continue
        parts = [p.strip() for p in line.split('|')]
        if len(parts) >= 3 and parts[1] == sex and parts[2] == 'train-clean-100': ids.append(parts[0])
    return ids

if REFERENCE.startswith('http'):
    urllib.request.urlretrieve(REFERENCE, '/content/ref_download'); audio = to16k('/content/ref_download')
else:
    if ':' in REFERENCE:
        sex, idx = REFERENCE.split(':'); spk = speakers_by_sex(sex.upper())[int(idx)]
    else:
        spk = REFERENCE
    files = sorted(glob.glob(f'{CORPUS}/{spk}/*/*.flac'))
    assert files, f'no audio found for speaker {spk}: re-run the corpus cell above'
    pieces, total = [], 0
    for f in files:
        a = to16k(f); pieces.append(a); total += len(a)
        if total >= 30 * 16000: break
    audio = torch.cat(pieces)
    print('reference speaker id:', spk)
sf.write(ref, audio.numpy(), 16000)
print('reference saved:', ref, f'({len(audio)/16000:.0f} s)')
from IPython.display import Audio, display
display(Audio(ref))  # listen: this is the voice being learned

In [ ]:
# Make training pairs (QuickVC converts the corpus into the target voice).
# Pairs live on Colab's local disk for speed and are mirrored to Drive so a new session can restore them.
import os, subprocess
data = '/content/data'
drive_data = f'{RUN}/data'
if os.path.exists(f'{data}/train'):
    print('pairs already on local disk')
elif os.path.exists(f'{drive_data}/train'):
    print('restoring pairs from Drive to local disk (a few minutes)...')
    subprocess.run(['rsync', '-a', '--info=progress2', f'{drive_data}/', f'{data}/'], check=True)
else:
    !cd /content/work/train && python make_pairs.py --src /content/corpus/LibriSpeech --ref "{ref}" --out "{data}" --hours {HOURS_OF_SPEECH} --device cuda
    print('mirroring pairs to Drive...')
    subprocess.run(['rsync', '-a', f'{data}/', f'{drive_data}/'], check=True)
print('pairs ready:', sum(len(os.listdir(f'{data}/{s}')) // 2 for s in ('train', 'val', 'dev')))

In [ ]:
# Training config
import json
cfg = json.load(open('/content/work/train/config_template.json'))
cfg['data']['dir'] = data
cfg['test_dir'] = '/content/work/LLVC/test_wavs'
json.dump(cfg, open(f'{RUN}/config.json', 'w'), indent=2)
print('config written')

In [ ]:
# Train (resumes automatically from the newest checkpoint in Drive)
!cd /content/work/train && python train_llvc.py --dir "{RUN}" --steps {TRAIN_STEPS} --device cuda

In [ ]:
# Export the voice file Voxel uses
import glob, os
ck = max(glob.glob(f'{RUN}/checkpoints/G_*.pth'), key=lambda p: int(os.path.basename(p).split('_')[1].split('.')[0]))
out = f'{ROOT}/voices/{VOICE_NAME}.onnx'
os.makedirs(os.path.dirname(out), exist_ok=True)
!cd /content/work/train && python export_onnx.py --checkpoint "{ck}" --config "{RUN}/config.json" --out "{out}" --chunk-factor 4
print('DONE — download these two files from Drive/Voxel/voices and send them to me:')
print(out); print(out.replace('.onnx', '.json'))